# BRCA internal val — held-out `rep1`, every finished weight

Xenium BRCA has two sections. `rep1` is fold 0 and is the only slide drawn. `rep2` is fold 1.

Each 2026-11-17 arm is the fold-0 checkpoint (`hce_group_fold_0.pt`), the weights that held `rep1` out. Their selected `best_mlp_gpu.pt` is fold 4, which trained on both BRCA slides, so these maps are not that checkpoint. A8 is the older deployment: its selected file is already fold 0, so the existing tree is the same held-out section.

`gate_l12_balanced` is still inside fold 0 and is not listed. This notebook only reads predictions.

```bash
CUDA_DEVICE=1 bash code/Internal_val/demo_uni2_axisfix_brca.sh predict
```


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

REPO = Path("/home/lingyu/ssd2/Python/Hist2Pheno")
if not (REPO / "code").is_dir():
    REPO = Path("/nobackup2/users/lingyu/Python/Hist2Pheno")
sys.path.insert(0, str(REPO / "code" / "Internal_val"))

import internal_val as iv

PAN = REPO / "data" / "PanCancer"
DATASET = "BRCA"
SAVE_FIGURES = False
iv.FOLD = 0

# A8 selected checkpoint is fold 0. The axisfix rows are fold-0 OOF trees
# (pred root ends in _brca_fold0/oof), not the fold-4 best_mlp.
MODELS = [
    (
        "A8",
        "older deployment: gate + balanced L12, fold 0",
        PAN / "pred_uni2_A8_l12_balanced_paired_v2_seed42_deployment",
        PAN / "result_uni2_A8_l12_balanced_paired_v2_seed42" / "cross_dataset_cv" / "A8_l12_balanced" / "best_mlp_gpu.pt",
    ),
    ("spatial_k8", "trunk: adjacent + k=8 mean", None, None),
    ("spatial_k8_gate", "trunk + spatial gate", None, None),
    ("l12_links_1", "trunk, L12 link weights = 1", None, None),
    ("l12_links_2", "trunk, L12 link weights = 2", None, None),
    ("l12_balanced", "trunk + balanced L12", None, None),
    ("focal_gamma1", "trunk + focal γ=1, class weights kept", None, None),
    ("balanced_sampler", "trunk + sampler, no class weight", None, None),
    ("legacy_nospatial", "old loss, no neighbors", None, None),
    ("adjacent_nospatial", "adjacent loss, no neighbors", None, None),
    ("l12_balanced_nospatial", "balanced L12, no neighbors", None, None),
]


def use_model(name, pred_root, ckpt):
    if pred_root is None:
        result = PAN / f"result_uni2_ablation_{name}_new1066axisfix_paired_v2_seed42"
        pred_root = PAN / f"pred_uni2_ablation_{name}_new1066axisfix_paired_v2_seed42_brca_fold0" / "oof"
        ckpt = result / "cross_dataset_cv" / name / "hce_group_fold_0.pt"
    iv.PRED_ROOT = pred_root
    iv.CKPT = ckpt
    iv.FOLD_PLAN = ckpt.parents[2] / "fold_plan.csv"
    iv.METRICS = pred_root / "metrics_per_section.csv"
    iv.FIG_DIR = pred_root / "internal_val_figures"
    return iv.viz_samples(DATASET)


name0, note0, pred0, ckpt0 = MODELS[0]
viz0 = use_model(name0, pred0, ckpt0)
print(f"{DATASET} fold {iv.FOLD}: {len(MODELS)} models, drawing {viz0}")
for name, note, pred_root, ckpt in MODELS:
    root = pred_root if pred_root is not None else PAN / f"pred_uni2_ablation_{name}_new1066axisfix_paired_v2_seed42_brca_fold0" / "oof"
    pred = root / "matched" / DATASET / "rep1" / "rep1_matched_pred.h5ad"
    print(f"{'ok ' if pred.is_file() else 'MISSING'}  {name}  {note}")


In [ ]:
for name, note, pred_root, ckpt in MODELS:
    viz = use_model(name, pred_root, ckpt)
    display(Markdown(f"## {name} \u2014 {note}"))
    missing = [s for s in viz if not iv.pred_paths(DATASET, s)["pred"].is_file()]
    if missing:
        print("missing", ", ".join(missing), "\u2014 CUDA_DEVICE=1 bash code/Internal_val/demo_uni2_axisfix_brca.sh predict", name if name != "A8" else "(A8 tree is the older deployment)")
        continue
    met = iv.metrics_fold0(DATASET)
    if not met.empty and "macro_auc_L2_cells" in met.columns:
        show = met.loc[met["sample"].astype(str).isin(viz), ["sample", "n_matched", "macro_auc_L2_cells"]]
        display(show)
    for sample in viz:
        fig = iv.plot_spatial_gt_pred(DATASET, sample, "L2", save=SAVE_FIGURES, title=f"{name}  {sample}")
        display(fig)
        plt.close(fig)
        fig = iv.plot_composition(DATASET, sample, "L2", save=SAVE_FIGURES, title=f"{name}  {sample}")
        display(fig)
        plt.close(fig)



In [ ]:
# GT and composition for every model are drawn in the cell above.

